<a href="https://colab.research.google.com/github/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/blob/main/ML_LAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# Session 1: FIND-S Algorithm
import csv
import urllib.request

def find_s_algorithm(url_or_file):
    # If it's a URL, download and read it
    if url_or_file.startswith('http://') or url_or_file.startswith('https://'):
        with urllib.request.urlopen(url_or_file) as response:
            lines = [line.decode('utf-8') for line in response.readlines()]
            reader = list(csv.reader(lines))
    else:
        with open(url_or_file, 'r') as file:
            reader = list(csv.reader(file))

    header = reader[0]
    data = reader[1:]

    # Initialize hypothesis with the most specific hypothesis: ['0', '0', ...]
    num_attributes = len(data[0]) - 1
    hypothesis = ['0'] * num_attributes

    print("Initial Hypothesis:", hypothesis)
    print("-" * 50)

    for i, row in enumerate(data):
        features = row[:-1]
        target = row[-1]

        # Consider only positive training instances
        if target.strip().lower() in ['yes', '1', 'positive']:
            if hypothesis == ['0'] * num_attributes:
                # First positive example initializes the hypothesis
                hypothesis = features.copy()
            else:
                for j in range(num_attributes):
                    if hypothesis[j] != features[j]:
                        hypothesis[j] = '?'
            print(f"Instance {i + 1} ({target}): {features}")
            print(f"Hypothesis after instance {i + 1}: {hypothesis}\n")
        else:
            print(f"Instance {i + 1} ({target}): {features} (Ignored)\n")

    print("-" * 50)
    print("Most Specific Hypothesis Found:")
    print(hypothesis)

# Run Find-S using your GitHub data
github_training_data_url = 'https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/training_data.csv'
find_s_algorithm(github_training_data_url)

Initial Hypothesis: ['0', '0', '0', '0', '0', '0']
--------------------------------------------------
Instance 1 (Yes): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Hypothesis after instance 1: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']

Instance 2 (Yes): ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same']
Hypothesis after instance 2: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']

Instance 3 (No): ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] (Ignored)

Instance 4 (Yes): ['Sunny', 'Warm', 'High', 'Strong', 'Cool', 'Change']
Hypothesis after instance 4: ['Sunny', 'Warm', '?', 'Strong', '?', '?']

--------------------------------------------------
Most Specific Hypothesis Found:
['Sunny', 'Warm', '?', 'Strong', '?', '?']


In [9]:
# Session 2: Candidate-Elimination Algorithm
import csv
import urllib.request

def candidate_elimination(url_or_file):
    # If it's a URL, download and read it
    if url_or_file.startswith('http://') or url_or_file.startswith('https://'):
        with urllib.request.urlopen(url_or_file) as response:
            lines = [line.decode('utf-8') for line in response.readlines()]
            reader = list(csv.reader(lines))
    else:
        with open(url_or_file, 'r') as file:
            reader = list(csv.reader(file))

    header = reader[0]
    data = reader[1:]

    num_attributes = len(data[0]) - 1

    # Initialize Specific and General Boundaries
    specific_h = ['0'] * num_attributes
    general_h = [['?' for _ in range(num_attributes)]]

    # Initialize S with the first positive example
    for row in data:
        if row[-1].strip().lower() in ['yes', '1', 'positive']:
            specific_h = row[:-1].copy()
            break

    print("Initial Specific Boundary (S):", specific_h)
    print("Initial General Boundary (G):", general_h)
    print("-" * 60)

    for idx, row in enumerate(data):
        features = row[:-1]
        label = row[-1].strip().lower()

        if label in ['yes', '1', 'positive']:
            # Positive Example
            # Remove any hypothesis from G that is inconsistent with features
            general_h = [
                g for g in general_h
                if all(g[i] == '?' or g[i] == features[i] for i in range(num_attributes))
            ]

            # Generalize S to cover features
            for i in range(num_attributes):
                if specific_h[i] != features[i]:
                    specific_h[i] = '?'

        else:
            # Negative Example
            # Specialize G to exclude features while remaining consistent with S
            new_general_h = []
            for g in general_h:
                if not all(g[i] == '?' or g[i] == features[i] for i in range(num_attributes)):
                    new_general_h.append(g)
                else:
                    for i in range(num_attributes):
                        if g[i] == '?' and specific_h[i] != '?' and specific_h[i] != features[i]:
                            spec = g.copy()
                            spec[i] = specific_h[i]
                            new_general_h.append(spec)

            # Filter duplicates and redundant hypotheses
            general_h = [
                h for i, h in enumerate(new_general_h)
                if h not in new_general_h[:i]
            ]

        print(f"Step {idx + 1} (Instance: {features} -> {row[-1]}):")
        print("  S:", specific_h)
        print("  G:", general_h)
        print("-" * 60)

    print("Final Version Space:")
    print("Specific Boundary (S):", specific_h)
    print("General Boundary (G):", general_h)

# Run Candidate-Elimination using your GitHub data
github_training_data_url = 'https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/training_data.csv'
candidate_elimination(github_training_data_url)

Initial Specific Boundary (S): ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
Initial General Boundary (G): [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 1 (Instance: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same'] -> Yes):
  S: ['Sunny', 'Warm', 'Normal', 'Strong', 'Warm', 'Same']
  G: [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 2 (Instance: ['Sunny', 'Warm', 'High', 'Strong', 'Warm', 'Same'] -> Yes):
  S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G: [['?', '?', '?', '?', '?', '?']]
------------------------------------------------------------
Step 3 (Instance: ['Rainy', 'Cold', 'High', 'Strong', 'Warm', 'Change'] -> No):
  S: ['Sunny', 'Warm', '?', 'Strong', 'Warm', 'Same']
  G: [['Sunny', '?', '?', '?', '?', '?'], ['?', 'Warm', '?', '?', '?', '?'], ['?', '?', '?', '?', '?', 'Same']]
------------------------------------------------------------
Step 4 (Insta

In [11]:
# Session 3: Decision Tree using ID3 Algorithm
import pandas as pd
import numpy as np

# 1. Calculate Entropy of a dataset
def entropy(target_col):
    elements, counts = np.unique(target_col, return_counts=True)
    entropy_val = 0
    for i in range(len(elements)):
        probability = counts[i] / np.sum(counts)
        entropy_val -= probability * np.log2(probability)
    return entropy_val

# 2. Calculate Information Gain for a feature
def information_gain(data, split_attribute_name, target_attribute_name="PlayTennis"):
    total_entropy = entropy(data[target_attribute_name])

    vals, counts = np.unique(data[split_attribute_name], return_counts=True)
    weighted_entropy = 0

    for i in range(len(vals)):
        sub_data = data[data[split_attribute_name] == vals[i]]
        prob = counts[i] / np.sum(counts)
        weighted_entropy += prob * entropy(sub_data[target_attribute_name])

    info_gain = total_entropy - weighted_entropy
    return info_gain

# 3. ID3 Algorithm Recursive Function
def id3(data, original_data, features, target_attribute_name="PlayTennis", parent_node_class=None):
    # Case 1: If target values are homogeneous (all 'Yes' or all 'No')
    if len(np.unique(data[target_attribute_name])) <= 1:
        return np.unique(data[target_attribute_name])[0]

    # Case 2: If dataset is empty, return target feature value in original dataset
    elif len(data) == 0:
        return np.unique(original_data[target_attribute_name])[
            np.argmax(np.unique(original_data[target_attribute_name], return_counts=True)[1])
        ]

    # Case 3: If feature set is empty, return parent node class
    elif len(features) == 0:
        return parent_node_class

    # Standard Case: Build tree recursively
    else:
        # Determine majority class for current node
        counts = np.unique(data[target_attribute_name], return_counts=True)[1]
        parent_node_class = np.unique(data[target_attribute_name])[np.argmax(counts)]

        # Select feature with maximum Information Gain
        item_values = [information_gain(data, feature, target_attribute_name) for feature in features]
        best_feature_index = np.argmax(item_values)
        best_feature = features[best_feature_index]

        # Create tree root structure
        tree = {best_feature: {}}

        # Remove best feature from feature list
        features = [i for i in features if i != best_feature]

        # Recursively grow tree for each branch/value
        for value in np.unique(data[best_feature]):
            sub_data = data[data[best_feature] == value]
            subtree = id3(sub_data, original_data, features, target_attribute_name, parent_node_class)
            tree[best_feature][value] = subtree

        return tree

# 4. Predict function for new unseen sample
def predict(query, tree, default='Yes'):
    for key in list(query.keys()):
        if key in list(tree.keys()):
            try:
                result = tree[key][query[key]]
            except KeyError:
                return default

            result = tree[key][query[key]]
            if isinstance(result, dict):
                return predict(query, result)
            else:
                return result

# --- Main Execution ---
if __name__ == "__main__":
    # GitHub Raw URL for play_tennis.csv
    url = "https://raw.githubusercontent.com/Priyanka-c0de/ITA0606-ML-Lab-Experiments-/main/datasets/play_tennis.csv"

    # Load dataset directly from URL
    df = pd.read_csv(url)

    # Clean data (Remove 'Day' or similar index columns if present)
    if 'Day' in df.columns:
        df = df.drop(columns=['Day'])
    if 'Humidity' in df.columns:
        df = df[df['Humidity'] != '???']

    # Features & Target Column
    features = [col for col in df.columns if col != 'PlayTennis']

    # Build Decision Tree
    decision_tree = id3(df, df, features)

    print("Constructed Decision Tree Structure:")
    import pprint
    pprint.pprint(decision_tree)
    print("-" * 50)

    # Predict a new sample
    new_sample = {
        'Outlook': 'Sunny',
        'Temperature': 'Cool',
        'Humidity': 'High',
        'Wind': 'Strong'
    }

    prediction = predict(new_sample, decision_tree)
    print("New Sample:", new_sample)
    print("Classification Result for New Sample (PlayTennis):", prediction)

Constructed Decision Tree Structure:
{'Outlook': {'Overcast': 'Yes',
             'Rain': {'Wind': {'Light': 'Yes', 'Strong': 'No'}},
             'Sunny': {'Temperature': {'Cool': 'Yes',
                                       'Hot': 'No',
                                       'Mild': 'Yes'}}}}
--------------------------------------------------
New Sample: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Classification Result for New Sample (PlayTennis): Yes
